In [1]:
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

I0000 00:00:1793355605.435859  259183 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793355605.436440  259183 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1793355605.493313  259183 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793355607.070740  259183 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

In [2]:
from pathlib import Path

# Use a stable project root so the notebook works from any working directory.
project_root = Path.cwd()
if not (project_root / 'Data' / 'Alzheimerr' / 'train_set').exists():
    project_root = project_root.parent

train_dir = project_root / 'Data' / 'Alzheimerr' / 'train_set'
test_dir = project_root / 'Data' / 'Alzheimerr' / 'test_set'

if not train_dir.exists() or not test_dir.exists():
    raise FileNotFoundError(f"Dataset folders not found. Checked: {train_dir} and {test_dir}")

train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.1,
    validation_split=0.2
)
validation_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)
test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    str(train_dir),
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical',
    subset='training',
    seed=42,
    shuffle=True
)

val_gen = validation_datagen.flow_from_directory(
    str(train_dir),
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42
)

test_gen = test_datagen.flow_from_directory(
    str(test_dir),
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

print(f"Train samples: {train_gen.samples}")
print(f"Validation samples: {val_gen.samples}")
print(f"Test samples: {test_gen.samples}")


Found 5760 images belonging to 4 classes.


Found 1440 images belonging to 4 classes.
Found 800 images belonging to 4 classes.
Train samples: 5760
Validation samples: 1440
Test samples: 800


In [ ]:
def build_cnn_model():
    model = models.Sequential([
        layers.Input(shape=(128, 128, 3)),

        # Block 1
        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D((2, 2)),

        # Block 2
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D((2, 2)),

        # Block 3
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dense(4, activation='softmax')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


def train_cnn_model(model, train_gen, val_gen, epochs=40, patience=6):
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=patience,
            restore_best_weights=True,
            verbose=1
        )
    ]
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=epochs,
        callbacks=callbacks
    )
    return history


def evaluate_cnn_model(model, test_gen):
    test_loss, test_accuracy = model.evaluate(test_gen, verbose=1)
    return {
        'test_loss': test_loss,
        'test_accuracy': test_accuracy,
    }


def save_cnn_outputs(model, history, output_dir='../Outputs'):
    os.makedirs(output_dir, exist_ok=True)
    model_path = os.path.join(output_dir, 'alzheimer_cnn_model.h5')
    model.save(model_path)

    plt.figure()
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title('CNN Training Performance')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plot_path = os.path.join(output_dir, 'cnn_plot.png')
    plt.savefig(plot_path)
    plt.close()

    print(f"Model saved to: {model_path}")
    print(f"Plot saved to: {plot_path}")
    return model_path, plot_path


def predict_single_image(model, image_path, class_names=None, target_size=(128, 128)):
    if class_names is None:
        class_names = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']

    img = tf.keras.preprocessing.image.load_img(image_path, target_size=target_size)
    img_array = tf.keras.preprocessing.image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    prediction = model.predict(img_array, verbose=0)
    predicted_class = int(np.argmax(prediction))
    return class_names[predicted_class], prediction


In [ ]:
model = build_cnn_model()
model.summary()


E0000 00:00:1793355608.033056  259183 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 128, 128, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 64, 64, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 32, 32, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 32768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │       131,076 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 418,084 (1.59 MB)

 Trainable params: 418,084 (1.59 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
print("Training the CNN...")
history = train_cnn_model(model, train_gen, val_gen, epochs=40, patience=6)


Training the CNN...
Epoch 1/40


I0000 00:00:1793355662.595484  259183 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.


180/180 ━━━━━━━━━━━━━━━━━━━━ 91s 499ms/step - accuracy: 0.2465 - loss: 1.3874 - val_accuracy: 0.2500 - val_loss: 1.3863
Epoch 2/40
180/180 ━━━━━━━━━━━━━━━━━━━━ 141s 493ms/step - accuracy: 0.2399 - loss: 1.3864 - val_accuracy: 0.2500 - val_loss: 1.3863
Epoch 3/40
180/180 ━━━━━━━━━━━━━━━━━━━━ 143s 498ms/step - accuracy: 0.2438 - loss: 1.3864 - val_accuracy: 0.2500 - val_loss: 1.3863
Epoch 4/40
180/180 ━━━━━━━━━━━━━━━━━━━━ 96s 531ms/step - accuracy: 0.2420 - loss: 1.3864 - val_accuracy: 0.2500 - val_loss: 1.3863
Epoch 5/40
180/180 ━━━━━━━━━━━━━━━━━━━━ 95s 528ms/step - accuracy: 0.2491 - loss: 1.3864 - val_accuracy: 0.2500 - val_loss: 1.3863
Epoch 6/40
180/180 ━━━━━━━━━━━━━━━━━━━━ 92s 509ms/step - accuracy: 0.2441 - loss: 1.3864 - val_accuracy: 0.2500 - val_loss: 1.3863
Epoch 7/40
180/180 ━━━━━━━━━━━━━━━━━━━━ 91s 506ms/step - accuracy: 0.2462 - loss: 1.3864 - val_accuracy: 0.2500 - val_loss: 1.3863
Epoch 7: early stopping
Restoring model weights from the end of the best epoch: 1.


In [ ]:
save_cnn_outputs(model, history, output_dir='../Outputs')
print("File successfully created in Outputs folder!")


In [ ]:
import os
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model

# 1. LOAD THE MODEL
model_path = '../Outputs/alzheimer_cnn_model.h5'
model = load_model(model_path)
print("Model loaded successfully!")

# 2. Automatically find the first image
folder_path = "../Data/Alzheimerr/test_set/MildDemented/"
all_images = os.listdir(folder_path)
first_image_name = all_images[0]
img_path = os.path.join(folder_path, first_image_name)
print(f"Testing with image: {img_path}")

# 3. Predict using reusable function
class_names = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']
predicted_class, prediction = predict_single_image(model, img_path, class_names=class_names)
print(f"The model predicts this image is: {predicted_class}")
print(f"Prediction probabilities: {prediction[0]}")


Model loaded successfully!
Testing with image: ../Data/Alzheimerr/test_set/MildDemented/MildDemented (1).jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
The model predicts this image is: VeryMildDemented


In [ ]:
# Save the CNN model file
model.save('../Outputs/alzheimer_cnn_model.h5')

# Save the CNN accuracy plot
plt.figure()
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('CNN Training Performance')
plt.legend()
plt.savefig('../Outputs/cnn_plot.png')

print("CNN Model and Plot saved to Outputs folder!")